# FUNCTIONS LIBRARY

Core ingestion and processing library for environmental sensor data collected across seven occupied dwellings in England (2019–2022). This module provides 17 functions covering the full data pipeline:

- `load_correction_factors` — loads pre-computed colocation calibration coefficients from CSV
- `ingest_aq110a` — reads Eltek AQ110A multi-pollutant logger exports and applies unit conversions, correction factors, and PM quality filtering
- `aq110a_no2_co1_baseline` — sets a 5th-percentile zero baseline for CO and NO2 channels
- `ingest_u12_012` — reads ONSET HOBO bathroom logger and applies colocation corrections
- `ingest_mx1102` — reads ONSET HOBO bedroom logger and applies colocation corrections
- `ingest_ux90_001m` — reads a single UX90-001M reed-switch event CSV into an event dictionary
- `ingest_ux90_001m_group_new` — reads multiple UX90-001M reed-switch event CSVs into event dictionaries
- `ingest_ux90_006_group` — reads UX90-006 occupancy event files
- `co2_correction_unoccupied` — applies a piecewise-linear NDIR drift correction anchored to manually identified unoccupied periods
- `co2_col_list` — extracts indoor CO2 column names from a DataFrame
- `ingest_ceda_weather_data` — reads CEDA Met Office hourly weather CSVs and resamples to 5-minute resolution
- `round_time` — rounds datetime objects up, down, or to the nearest delta
- `rooms_variables_dict` — builds a room-to-variables mapping dictionary
- `all_events_df` — concatenates event dictionaries into a single chronological DataFrame
- `event_durations` — computes elapsed minutes between consecutive events
- `event_list_function` — returns a flat list of event names
- `sum_sound_calculation_df` — performs logarithmic (energy) averaging of one-third-octave sound levels
- `rpi_noise_ingest` — ingests Raspberry Pi PHEUCLio noise CSVs and returns structured indoor/outdoor DataFrames

## Import Packages

In [ ]:
import numpy as np  # Numerical computing and array operations
import pandas as pd  # DataFrame manipulation and CSV I/O
import glob  # Unix-style pathname pattern expansion
import os  # Operating system interface for directory walking
import pickle
import datetime  # Date and time arithmetic
import csv  # Low-level CSV reading and writing
import statistics  # Standard-library descriptive statistics
import fnmatch  # Unix filename pattern matching (used in noise ingest)
import math  # Core mathematical functions
from time import sleep  # Pause execution for a given number of seconds
# Matplotlib
import matplotlib.pylab as plt  # MATLAB-style plotting interface
from matplotlib.collections import PathCollection  # Used for scatter-plot collections
from matplotlib import rc  # Runtime configuration for matplotlib
# Seaborn
import seaborn as sns  # High-level statistical visualisation built on matplotlib
# Scipy
import scipy  # Scientific computing library
from scipy.stats import linregress  # Ordinary least-squares linear regression
from scipy.optimize import curve_fit  # Non-linear least-squares curve fitting
from scipy import stats  # Statistical functions and distributions
from scipy.stats import shapiro  # Shapiro-Wilk normality test
from scipy.stats import ks_2samp  # Two-sample Kolmogorov-Smirnov test
from scipy.stats import mannwhitneyu  # Mann-Whitney U non-parametric test
from scipy.signal import find_peaks  # Peak detection in 1-D arrays
from scipy.stats import spearmanr  # Spearman rank-order correlation
# Iteration tools
import itertools  # Iterator building blocks (combinations, products, etc.)
# Allow iterate through operators
import operator as op  # Standard operators as callable functions
# Other
from pylab import figure, text, scatter, show  # Convenience pylab plotting functions
from functools import reduce  # Apply a function cumulatively to a sequence
from pathlib import Path  # Object-oriented filesystem paths
# from datetime import timedelta, datetime
from tqdm import tqdm  # Progress-bar wrapper for iterables
# Sklearn
from sklearn import datasets, linear_model  # Scikit-learn datasets and linear models
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error  # Regression metrics
from scipy.stats import iqr  # Interquartile range statistic

from statsmodels.stats.stattools import durbin_watson  # Durbin-Watson autocorrelation test statistic
# Annotate plots
from statannotations.Annotator import Annotator  # Statistical annotation for seaborn plots
print('Packages loaded.')  # Confirm all imports completed successfully

# Change fonts
plt.rcParams['mathtext.fontset'] = 'stix'  # Use STIX font set for mathematical text
plt.rcParams['font.family'] = 'STIXGeneral'  # Set default font family to STIXGeneral

## load_correction_factors

Loads pre-computed calibration offsets and slopes from `correction_factors.csv`, which is generated during the pre- and post-campaign colocation campaigns. The CSV maps each instrument reference (e.g. `32656_temperature`) to three correction terms: `baseline_correction` (additive offset applied first), `slope` (multiplicative scale factor), and `intercept` (additive offset applied after scaling). AQ110A instrument serial numbers are remapped from their Eltek `EI…` format to their numeric channel IDs. The result is stored in the module-level variable `correction_factors` so all ingestion functions can reference it without re-reading the file.

In [ ]:
def load_correction_factors():
    """
    Load instrument correction factors from the colocation CSV file.

    Reads ``correction_factors.csv`` located two directories above the calling
    script, constructs a lookup ``instrument_ref`` column by combining the
    instrument serial number (remapped from Eltek EI-codes to numeric IDs) with
    the measured variable name, and returns the resulting DataFrame for use by
    all ingestion functions.

    Parameters
    ----------
    None

    Returns
    -------
    correction_factors : pd.DataFrame
        DataFrame indexed by row number with columns including
        ``instrument_ref`` (str), ``baseline_correction`` (float),
        ``slope`` (float), and ``intercept`` (float).
    """

    correction_factors = pd.read_csv('../data/correction_factors/correction_factors.csv', skiprows = 0, index_col=0) # ingest
    correction_factors['sub_variable'] = correction_factors['sub_variable'].astype("string")  # Cast sub_variable to pandas StringDtype for reliable string operations
    correction_factors['instrument'] = correction_factors['sub_variable'].str.split('_',expand=True)[0]  # Extract leading instrument token before the first underscore
    correction_factors['instrument_ref'] = correction_factors['instrument']  # Initialise instrument_ref column as a copy of the raw instrument token
    aq110_col_rename_dict = {
      "EI130885": "32656",  # Map Eltek serial EI130885 to numeric channel ID 32656
      "EI130887": "32658",  # Map Eltek serial EI130887 to numeric channel ID 32658
      "EI130963": "34642",  # Map Eltek serial EI130963 to numeric channel ID 34642
      "EI130961": "34645",  # Map Eltek serial EI130961 to numeric channel ID 34645
      "EI130959": "34644",  # Map Eltek serial EI130959 to numeric channel ID 34644
      "EI130895": "32672",  # Map Eltek serial EI130895 to numeric channel ID 32672
      "EI130894": "32670",  # Map Eltek serial EI130894 to numeric channel ID 32670
      "EI130886": "32657",  # Map Eltek serial EI130886 to numeric channel ID 32657
      "EI130960": "34643",  # Map Eltek serial EI130960 to numeric channel ID 34643
      "EI130611": "30819",  # Map Eltek serial EI130611 to numeric channel ID 30819
      "EI130962": "34646",  # Map Eltek serial EI130962 to numeric channel ID 34646
      "EI130996": "36241",  # Map Eltek serial EI130996 to numeric channel ID 36241
      "EI130995": "36248",  # Map Eltek serial EI130995 to numeric channel ID 36248
      "EI130997": "36244",  # Map Eltek serial EI130997 to numeric channel ID 36244
      "EI130994": "36249",  # Map Eltek serial EI130994 to numeric channel ID 36249
      "EI130993": "36245",  # Map Eltek serial EI130993 to numeric channel ID 36245
      "EI130986": "35992",  # Map Eltek serial EI130986 to numeric channel ID 35992
      "EI130992": "36243",  # Map Eltek serial EI130992 to numeric channel ID 36243
      "EI130991": "36246",  # Map Eltek serial EI130991 to numeric channel ID 36246
    }
    #
    # Replace strings as dictionary above
    #
    for old, new in aq110_col_rename_dict.items():  # Iterate over each EI-code to numeric-ID mapping
        # print(old,new)  # Log the substitution being applied
        correction_factors['instrument_ref'] = correction_factors['instrument_ref'].str.replace(old, new, regex=False)  # Substitute EI serial with numeric ID using exact (non-regex) match
    correction_factors['instrument_ref'] = correction_factors['instrument_ref']+'_'+correction_factors['variable']  # Append variable name to form the full lookup key e.g. '32656_temperature'
    #
    # Return correction_factors df
    #
    return(correction_factors)  # Return completed correction factors DataFrame

correction_factors = load_correction_factors()  # Load correction factors into the global namespace at import time

## ingest_aq110a

Ingests data from the Eltek AQ110A multi-sensor CSV export. The AQ110A logs temperature, relative humidity, CO, CO2, PM1/PM2.5/PM10, NO2, and tVOC at 5-minute intervals. The raw CSV has a three-row header; the first data row contains channel labels which are combined with the caller-supplied `import_variables` list to construct unique column names. After parsing, the function: rounds timestamps to the nearest 5 minutes; replaces sentinel strings (`'No Data'`, `'High'`, `'Low'`, `'Open'`) with NaN; optionally linearly interpolates short gaps; applies the three-term colocation correction (baseline offset, slope, intercept) from the global `correction_factors` DataFrame; optionally converts NO2 from ppm to µg/m³ (factor 1912.5) and CO from ppm to µg/m³ (factor 1164.2); renames columns to the caller-supplied list; and sets PM values to NaN where the co-located relative humidity exceeds 85 % to avoid hygroscopic mass overestimation.

In [ ]:
def ingest_aq110a(path
                  , import_variables
                  , column_names
                  , convert_no2
                  , convert_co
                  , input_interpolate
                  , input_iterpolate_limit
                  ):
    """
    Ingest and correct Eltek AQ110A multi-sensor CSV data.

    Reads a raw AQ110A CSV export (3-row header), constructs unambiguous column
    names by combining channel labels with caller-supplied variable names, parses
    and rounds timestamps, replaces sentinel error strings with NaN, optionally
    interpolates short gaps, applies colocation correction factors, optionally
    converts NO2 and CO from ppm to µg/m³, and masks PM values where RH > 85 %.

    Parameters
    ----------
    path : str
        Full filesystem path to the AQ110A CSV export file.
    import_variables : list of str
        Variable-role labels (e.g. ``['temperature', 'humidity', 'co2', ...]``)
        aligned positionally with the CSV channel columns.
    column_names : list of str
        Final output column names applied after all corrections.
    convert_no2 : bool
        If ``True``, multiply NO2 columns by 1912.5 to convert ppm → µg/m³.
    convert_co : bool
        If ``True``, multiply CO (non-CO2) columns by 1164.2 to convert ppm → µg/m³.
    input_interpolate : bool
        If ``True``, apply forward linear interpolation to fill short gaps.
    input_iterpolate_limit : int
        Maximum consecutive NaN values to fill when interpolating.

    Returns
    -------
    eltek : pd.DataFrame
        Datetime-indexed DataFrame (5-minute resolution) containing corrected
        and unit-converted sensor readings.
    """
    
    #     convert_no2 = convert from ppm to ug/m3?
    #     convert_co = convert from ppm to ug/m3?
        
    #     Convert NO2 and CO from PPM into ug/m3
    #     Both NO2 and CO measured in PPM parts per million
    #     CO      1 ppm = 1164.2  ug/m3
    #             1 ppm = 1.1642  mg/m3
        
    #     NO2     1 ppb = 1.9125  µg m-3
    #             1 ppm = 1912.5  ug/m3
    
    # Start of function
    print('')  # Blank line for readability in console output
    print('---------------------------------------------')  # Visual separator for this ingest block
    print('Ingesting AQ110a data from', path)  # Log the source file path
    
    eltek = pd.read_csv(path, skiprows = 3) # ingest
    new_col_list = []  # Initialise list to accumulate constructed column names
    temp_col_list = eltek.iloc[0].tolist()  # Row 0 contains per-channel label strings from the AQ110A export
    temp_col_list = np.char.replace(temp_col_list, 'TX Serial Number', 'date')  # Rename the timestamp label to 'date' for consistent indexing
    eltek_col_lis = import_variables  # Alias the caller-supplied variable-role list
    
    for combined_1, combined_2 in zip(temp_col_list, eltek_col_lis):  # Pair each channel label with its variable role
        # print(combined_1, combined_2)
        new_name = combined_1 + '_' + combined_2  # Concatenate channel label and role to form a unique column name
        new_col_list.append(new_name)  # Accumulate the constructed name
        # print(new_col_list)
    
    new_col_list = np.char.replace(new_col_list, 'date_', 'date')  # Strip the redundant 'date_' prefix from the timestamp column name
    eltek.columns = new_col_list  # Apply the constructed column names to the raw DataFrame
    eltek = eltek.iloc[2:]  # Drop the two header rows (channel labels and units) retained after skiprows
    eltek['date'] = pd.to_datetime(eltek['date'], errors='raise', dayfirst=True) # date to datetime
    eltek['date'] = eltek['date'].dt.round('5min') # round date to nearest 'rounding' minutes
    eltek = eltek.replace('No Data', np.nan) # No data to nan
    eltek = eltek.replace('High', np.nan) # High to nan
    eltek = eltek.replace('Low', np.nan) # Low to nan
    eltek = eltek.replace('Open', np.nan) # Open to nan 
    eltek = eltek.drop_duplicates(subset=['date'])  # Remove any duplicate timestamps arising from rounding to 5 minutes
    eltek = eltek.set_index(eltek['date'])  # Set the datetime column as the DataFrame index
    del eltek['date']  # Remove the now-redundant 'date' column to avoid duplication
    eltek = eltek.apply(pd.to_numeric) # change all to float
    
    # Interpolate
    if input_interpolate == True:  # Only interpolate if the caller requested it
        eltek = eltek.interpolate(method='linear', limit = input_iterpolate_limit, limit_direction='forward')  # Forward-fill short gaps using linear interpolation up to the specified limit

    # Correction
    print('')  # Blank line before correction log output
    print('Correction')  # Section header for correction logging
    print('')  # Blank line after section header
    #
    for col in eltek:  # Iterate over every column in the ingested DataFrame
        print('-----------------------')  # Visual separator per column
        print(col)  # Log the column name being corrected
        for correction_item in ['baseline_correction', 'slope', 'intercept']:  # Apply all three correction terms in sequence
            print(correction_item)  # Log which correction term is being applied
            print('')  # Blank line for readability
            correction_calc = correction_factors[correction_item].loc[correction_factors['instrument_ref'] == str(col)]  # Look up this column's correction value from the global correction_factors DataFrame
            correction_calc = correction_calc.values  # Extract as a NumPy array
            correction_calc_is_empty = correction_calc.size == 0 # new variable to check if correction calc is 0
            print(correction_calc_is_empty)  # Log whether a correction entry was found
            if correction_calc_is_empty == True:  # No entry found: use identity correction values
                if correction_item == 'baseline_correction':  # Default baseline offset is zero (no shift)
                    correction_calc = 0

                elif correction_item == 'slope':  # Default slope is one (no scaling)
                    correction_calc = 1

                elif correction_item == 'intercept':  # Default intercept is zero (no shift)
                    correction_calc = 0
            else:
                # Extract scalar value from array
                correction_calc = correction_calc[0]  # Unpack the single-element array to a scalar
            #
            correction_calc = float(correction_calc)  # Ensure the correction value is a Python float for arithmetic
            print(type(correction_calc))  # Log the type for debugging
            print(correction_calc)  # Log the numeric correction value
            #
            #
            if correction_item == 'baseline_correction':  # Apply additive baseline offset
                eltek[col] = eltek[col] + correction_calc

            elif correction_item == 'slope':  # Apply multiplicative scale correction
                eltek[col] = eltek[col] * correction_calc

            elif correction_item == 'intercept':  # Apply additive intercept after scaling
                eltek[col] = eltek[col] + correction_calc
    #
    # Convert co and no2
    #
    if convert_no2 == True:  # Convert NO2 from ppm to µg/m³ if requested
        for col in eltek:  # Check every column for an NO2 channel
            if 'no2' in col:  # Target only columns containing 'no2'
                eltek[col] = eltek[col] * 1912.5  # Apply NO2 unit conversion factor (1 ppm = 1912.5 µg/m³)
    if convert_co == True:  # Convert CO from ppm to µg/m³ if requested
        for col in eltek:  # Check every column for a CO channel
            if 'co' in col:  # Target columns containing 'co'
                if 'co2' in col:  # Exclude CO2 columns — only pure CO should be converted
                    print('')  # Skip CO2 columns silently
                else:
                    eltek[col] = eltek[col] * 1164.2  # Apply CO unit conversion factor (1 ppm = 1164.2 µg/m³)
    
    eltek.columns = column_names # change col names
    
    #
    # Ignore PM, where RH above 85\%
    #
    for col in eltek:  # Iterate over final output columns to find PM channels
        if any(x in col for x in ['1um' , '25um' , '100um']):  # Identify PM1, PM2.5, and PM10 columns by size-cut label
            # find name of room
            # print(col)
            room_name = col.split("_")[0]  # Extract the room prefix (e.g. 'living') from the column name
            # print(col.split("_")[0])
            # find all rows where RH>85% and turn to nan
            eltek.loc[eltek[room_name + '_humidity'] > 85, col] = np.nan  # Mask PM values where co-located RH exceeds 85 % (hygroscopic growth artefact)
    

    print('AQ110a data ingested')  # Confirm successful completion
    print('---------------------------------------------')  # Visual separator
    print('')  # Trailing blank line
    return(eltek)  # Return the corrected, datetime-indexed DataFrame

## aq110a_no2_co1_baseline

Sets the 5th-percentile as the zero baseline for CO and NO2 channels in a DataFrame that has already been ingested by `ingest_aq110a`. Electrochemical CO and NO2 sensors exhibit a non-zero quiescent output in clean air; subtracting the chosen low percentile (typically the 5th) of the full campaign record removes this systematic offset. Any values that become negative after the subtraction are clamped to zero, as negative gas concentrations are physically meaningless. The function modifies the DataFrame in place using `.loc` to avoid pandas chained-assignment warnings.

In [ ]:
def aq110a_no2_co1_baseline(input_dataframe, input_percentile):
    """
    Correct CO and NO2 baseline by subtracting a low-percentile reference value.

    For each column whose name contains ``'co1'`` or ``'no2'``, computes the
    specified percentile of the full record (ignoring NaN), subtracts it from
    all values, and clamps any resulting negatives to zero.

    Parameters
    ----------
    input_dataframe : pd.DataFrame
        DataFrame containing AQ110A channel data; must include columns with
        ``'co1'`` or ``'no2'`` in their names.
    input_percentile : float
        Percentile (0–100) used as the zero baseline, e.g. ``5`` for the 5th
        percentile.

    Returns
    -------
    None
        The input DataFrame is modified in place.
    """
    for col in input_dataframe.columns:  # Iterate over all columns in the DataFrame
        if any(x in col for x in ['co1', 'no2']):  # Target only CO and NO2 sensor channels
            if input_dataframe[col].isna().all():  # Guard: skip if the entire column is NaN (e.g. NO2 invalidated by RF interference)
                print('Baseline variable =', col, '— all-NaN column, skipping baseline correction')
                continue
            # Calculate the 5th percentile
            fifth_percentile = np.nanpercentile(input_dataframe[col], input_percentile)  # Compute the chosen percentile, ignoring NaN values
            # Log the baseline calculation
            print('Baseline variable =', col, '5th percentile =', fifth_percentile)  # Log the column name and computed baseline value
            # Adjust column values using .loc to avoid chained assignment
            input_dataframe.loc[:, col] = input_dataframe[col] - fifth_percentile  # Subtract the baseline from all values in the column
            # Set negative values to 0
            input_dataframe.loc[input_dataframe[col] < 0, col] = 0  # Clamp negative values to zero (physically impossible concentrations)

## ingest_u12_012

Ingests data from the ONSET HOBO U12-012 temperature, relative humidity, and light logger deployed in bathrooms. The U12-012 exports a CSV with separate date and time columns and a configurable number of header rows. The function concatenates date and time into a single datetime, optionally rounds to a specified interval (e.g. `'5min'`), optionally drops duplicate timestamps, and optionally sets the datetime as the DataFrame index. Colocation corrections are applied for temperature and humidity channels by looking up the `instrument_ref + '_temperature'` and `instrument_ref + '_humidity'` keys in the global `correction_factors` DataFrame and applying the three-term correction (baseline offset, slope, intercept).

In [ ]:
def ingest_u12_012(path
                   , skiprows
                   , column_names
                   , rounding
                   , drop_duplicates
                   , index_as_date
                   , instrument_ref
                   ):
    """
    Ingest and correct ONSET HOBO U12-012 bathroom logger CSV data.

    Reads a U12-012 CSV export, parses and optionally rounds the datetime,
    removes duplicate rows, and applies colocation corrections for temperature
    and humidity from the global ``correction_factors`` DataFrame.

    Parameters
    ----------
    path : str
        Full filesystem path to the U12-012 CSV export file.
    skiprows : int
        Number of header rows to skip when reading the CSV.
    column_names : list of str
        Column names to assign after reading (must include ``'date'`` and
        ``'time'`` as the first two entries).
    rounding : str
        Pandas offset alias for timestamp rounding, e.g. ``'5min'``.
    drop_duplicates : bool
        If ``True``, remove rows with duplicate ``'date'`` values after rounding.
    index_as_date : bool
        If ``True``, set the ``'date'`` column as the DataFrame index.
    instrument_ref : str
        Instrument reference string (e.g. ``'U12_bathroom'``) used to look up
        colocation correction factors.

    Returns
    -------
    u12_012 : pd.DataFrame
        Corrected DataFrame with datetime index (if ``index_as_date=True``) or
        a ``'date'`` column containing parsed timestamps.
    """
    
    # Start of function
    print('')  # Blank line for readability
    print('---------------------------------------------')  # Visual separator
    print('Ingesting U12-012 data from', path)  # Log the source file
    u12_012 = pd.read_csv(path, skiprows = skiprows)  # Read the raw CSV, skipping the specified number of header rows
    u12_012.columns = column_names  # Apply caller-supplied column names
    u12_012['date'] = pd.to_datetime(u12_012['date'] + ' ' + u12_012['time'], errors='raise', dayfirst=True)  # Combine separate date and time columns into a single datetime object
    del u12_012['time']  # Remove the now-redundant separate time column
    u12_012['date'] = u12_012['date'].dt.round(rounding) # rounding datetime
    
    # Drop duplicates
    if drop_duplicates == True:  # Only remove duplicates if the caller requested it
        u12_012 = u12_012.drop_duplicates(subset=['date'])  # Remove rows with duplicate rounded timestamps
    
    # Date as index
    if index_as_date == True:  # Only set date as index if the caller requested it
        u12_012 = u12_012.set_index(u12_012['date'])  # Set the datetime column as the DataFrame index
        del u12_012['date']  # Remove the now-redundant 'date' column
    #
    # Correction
    #
    for col in u12_012:  # Iterate over all columns
        for variable in ['temperature', 'humidity']:  # U12-012 carries only temperature and humidity (plus light, uncorrected)
            variable_to_find = instrument_ref + '_' + variable  # Construct the lookup key e.g. 'U12_bathroom_temperature'
            if variable in col:  # Only apply if the column name contains this variable type
                for correction_item in ['baseline_correction', 'slope', 'intercept']:  # Apply all three correction terms in sequence
                    print(correction_item)  # Log which correction term is being applied
                    print('')  # Blank line for readability
                    correction_calc = correction_factors[correction_item].loc[correction_factors['instrument_ref'] == str(variable_to_find)]  # Look up this variable's correction value
                    correction_calc = correction_calc.values  # Extract as a NumPy array
                    correction_calc_is_empty = correction_calc.size == 0  # Check whether any correction entry was found
                    print(correction_calc_is_empty)  # Log whether a correction was found
                    if correction_calc_is_empty == True:  # No entry: fall back to identity correction
                        if correction_item == 'baseline_correction':  # Identity baseline offset is zero
                            correction_calc = 0

                        elif correction_item == 'slope':  # Identity slope is one
                            correction_calc = 1

                        elif correction_item == 'intercept':  # Identity intercept is zero
                            correction_calc = 0
                    else:
                        # Extract scalar value from array
                        correction_calc = correction_calc[0]  # Unpack single-element array to a scalar
                    #
                    #
                    correction_calc = float(correction_calc)  # Ensure numeric type for arithmetic
                    print(type(correction_calc))  # Log the type for debugging
                    print(correction_calc)  # Log the numeric correction value
                    #
                    #
                    if correction_item == 'baseline_correction':  # Apply additive baseline offset
                        u12_012[col] = u12_012[col] + correction_calc

                    elif correction_item == 'slope':  # Apply multiplicative scale correction
                        u12_012[col] = u12_012[col] * correction_calc

                    elif correction_item == 'intercept':  # Apply additive intercept after scaling
                        u12_012[col] = u12_012[col] + correction_calc
    #
    #
    print('Missing data = \n', u12_012.isna().sum())  # Report the number of NaN values per column
    print('U12-012 data ingested')  # Confirm successful completion
    print('---------------------------------------------')  # Visual separator
    print('')  # Trailing blank line
    return(u12_012)  # Return the corrected DataFrame

## ingest_mx1102

Ingests data from the ONSET HOBO MX1102 CO2, temperature, and relative humidity logger deployed in bedrooms. The MX1102 exports a CSV with separate date and time columns. The function concatenates date and time, optionally rounds to a specified interval, optionally drops duplicate timestamps, applies the three-term colocation correction for temperature, humidity, and CO2 channels (looked up via `instrument_ref + '_<variable>'` in the global `correction_factors` DataFrame), and optionally sets the datetime as the DataFrame index. Note that CO2 NDIR drift correction is handled separately by `co2_correction_unoccupied` after ingestion.

In [ ]:
def ingest_mx1102(path
                  , skiprows
                  , column_names
                  , rounding
                  , drop_duplicates
                  , instrument_ref
                  , index_as_date
                  ):
    """
    Ingest and correct ONSET HOBO MX1102 bedroom CO2/temperature/humidity logger CSV data.

    Reads an MX1102 CSV export, parses and optionally rounds the datetime,
    removes duplicate rows, applies colocation corrections for temperature,
    humidity, and CO2, and optionally sets the datetime as the index.

    Parameters
    ----------
    path : str
        Full filesystem path to the MX1102 CSV export file.
    skiprows : int
        Number of header rows to skip when reading the CSV.
    column_names : list of str
        Column names to assign after reading (must include ``'date'`` and
        ``'time'`` as the first two entries).
    rounding : str
        Pandas offset alias for timestamp rounding, e.g. ``'5min'``.
    drop_duplicates : bool
        If ``True``, remove rows with duplicate ``'date'`` values after rounding.
    instrument_ref : str
        Instrument reference string (e.g. ``'MX1102_bedroom'``) used to look up
        colocation correction factors.
    index_as_date : bool
        If ``True``, set the ``'date'`` column as the DataFrame index.

    Returns
    -------
    mx1102 : pd.DataFrame
        Corrected DataFrame with datetime index (if ``index_as_date=True``) or
        a ``'date'`` column containing parsed timestamps.
    """
    
    # Start of function
    print('')  # Blank line for readability
    print('---------------------------------------------')  # Visual separator
    print('Ingesting MX1102 data from', path)  # Log the source file
    mx1102 = pd.read_csv(path, skiprows = skiprows)  # Read the raw CSV, skipping the specified number of header rows
    mx1102.columns = column_names  # Apply caller-supplied column names
    mx1102['date'] = pd.to_datetime(mx1102['date'] + ' ' + mx1102['time'], errors='raise', dayfirst=True)  # Combine separate date and time columns into a single datetime object
    del mx1102['time']  # Remove the now-redundant separate time column
    mx1102['date'] = mx1102['date'].dt.round(rounding)  # Round timestamps to the nearest specified interval
    
    # Drop duplicates
    if drop_duplicates == True:  # Only remove duplicates if the caller requested it
        mx1102 = mx1102.drop_duplicates(subset=['date'])  # Remove rows with duplicate rounded timestamps
    #
    # Correction
    #
    for col in mx1102:  # Iterate over all columns
        for variable in ['temperature', 'humidity', 'co2']:  # MX1102 carries temperature, humidity, and CO2
            variable_to_find = instrument_ref + '_' + variable  # Construct the lookup key e.g. 'MX1102_bedroom_co2'
            if variable in col:  # Only apply if this column name contains the variable type
                for correction_item in ['baseline_correction', 'slope', 'intercept']:  # Apply all three correction terms in sequence
                    print(correction_item)  # Log which correction term is being applied
                    print('')  # Blank line for readability
                    correction_calc = correction_factors[correction_item].loc[correction_factors['instrument_ref'] == str(variable_to_find)]  # Look up this variable's correction value
                    correction_calc = correction_calc.values  # Extract as a NumPy array
                    correction_calc_is_empty = correction_calc.size == 0  # Check whether any correction entry was found
                    print(correction_calc_is_empty)  # Log whether a correction was found
                    if correction_calc_is_empty == True:  # No entry: fall back to identity correction
                        if correction_item == 'baseline_correction':  # Identity baseline offset is zero
                            correction_calc = 0

                        elif correction_item == 'slope':  # Identity slope is one
                            correction_calc = 1

                        elif correction_item == 'intercept':  # Identity intercept is zero
                            correction_calc = 0
                    else:
                        # Extract scalar value from array
                        correction_calc = correction_calc[0]  # Unpack single-element array to a scalar
                    #
                    #
                    correction_calc = float(correction_calc)  # Ensure numeric type for arithmetic
                    print(type(correction_calc))  # Log the type for debugging
                    print(correction_calc)  # Log the numeric correction value
                    #
                    #
                    if correction_item == 'baseline_correction':  # Apply additive baseline offset
                        mx1102[col] = mx1102[col] + correction_calc

                    elif correction_item == 'slope':  # Apply multiplicative scale correction
                        mx1102[col] = mx1102[col] * correction_calc

                    elif correction_item == 'intercept':  # Apply additive intercept after scaling
                        mx1102[col] = mx1102[col] + correction_calc
    
    # Index as date
    if index_as_date == True:  # Only set date as index if the caller requested it
        mx1102 = mx1102.set_index(mx1102['date'])  # Set the datetime column as the DataFrame index
        del mx1102['date']  # Remove the now-redundant 'date' column
    
    
    print('Missing data = \n', mx1102.isna().sum())  # Report the number of NaN values per column
    print('MX1102 data ingested')  # Confirm successful completion
    print('---------------------------------------------')  # Visual separator
    print('')  # Trailing blank line
    return(mx1102)  # Return the corrected DataFrame

## ingest_ux90_001m_group_new

Ingests multiple UX90-001M event files into an event dictionary and simultaneously back-fills the events into a main time-series DataFrame. For each file/output-name pair, the function reads the event CSV, sets the datetime as the index, resamples to 5-minute resolution using backward fill (so the state recorded at an event timestamp is propagated back to the preceding 5-minute bin), and then inverts the state (0 ↔ 1) so that `1` represents *open* in the main DataFrame. The raw event table (with integer index, not datetime index) is retained in the returned dictionary for duration analysis by `event_durations`.

In [ ]:
def ingest_ux90_001m_group_new(input_file_list
                               , input_output_name_list
                               , input_skiprows
                               , input_main_dataframe
                               ):
    """
    Ingest multiple UX90-001M event CSV files into an event dictionary.

    For each file/name pair, reads the event CSV, sets the datetime index,
    back-fills into the main time-series DataFrame at 5-minute resolution
    (inverting the state so 1 = open), and stores the raw event table in a
    dictionary for downstream duration analysis.

    Parameters
    ----------
    input_file_list : list of str
        Ordered list of full filesystem paths to UX90-001M CSV export files.
    input_output_name_list : list of str
        Ordered list of output names (e.g. ``['kitchen_window', 'front_door']``)
        corresponding to each file in ``input_file_list``.
    input_skiprows : int
        Number of header rows to skip when reading each CSV.
    input_main_dataframe : pd.DataFrame
        Main time-series DataFrame (datetime-indexed at 5-minute resolution)
        into which the resampled event states are written in place.

    Returns
    -------
    event_dict : dict
        Dictionary mapping each output name to a DataFrame of raw event
        timestamps and state values (integer-indexed, not datetime-indexed).
    """
    # Start of function
    event_dict = {}    # Initialise a dictionary to accumulate event DataFrames
    for input_files, output_name in zip(input_file_list, input_output_name_list):  # Pair each file path with its designated output name
        
        print('')  # Blank line for readability
        print('---------------------------------------------')  # Visual separator
        print('Ingesting ux90_001m data from', input_files)  # Log the source file
        
        df = pd.read_csv(input_files, skiprows=input_skiprows, encoding='utf-8', engine='c')  # Read the event CSV with UTF-8 encoding and fast C parser
        df.columns = ['date', 'time', output_name]  # Assign date, time, and event-label column names
        df['date'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='raise', dayfirst=True)  # Merge date and time strings into a single datetime
        del df['time']  # Remove the now-redundant separate time column
        df = df.set_index(df['date'])  # index = date
        del df['date']  # Remove the now-redundant 'date' column after it has been set as the index
        
        # Put events into main df, backfilling for 5 mins
        input_main_dataframe[output_name] = df.resample('5min').bfill()  # Resample event timestamps to 5-minute bins, back-filling state from the event time
        # Change all 0 to 1 and 1 to 0
        input_main_dataframe[output_name] = 1 - input_main_dataframe[output_name]  # Invert state so that 1 represents open and 0 represents closed
        
        event_dict[output_name] = df.reset_index()  # Reset to integer index so the raw event table is suitable for duration analysis

    return(event_dict)  # Return dictionary of raw event DataFrames

## ingest_ux90_006_group

Ingests multiple UX90-006 occupancy event files into an occupancy dictionary. The UX90-006 is a passive infrared (PIR) occupancy logger that records a timestamp for each detected motion event. Unlike the UX90-001M window/door sensors, occupancy events are not back-filled into the main time-series DataFrame here — they are retained as raw event tables for downstream analysis. The function reports the number of non-unique timestamps per file (which can indicate sensor artefacts) and resets the index to a contiguous integer sequence before storing in the dictionary.

In [ ]:
def ingest_ux90_006_group(input_file_list
                          , output_name_list
                          , skiprows
                          ):
    """
    Ingest multiple UX90-006 occupancy event CSV files into an occupancy dictionary.

    For each file/name pair, reads the occupancy event CSV, merges date and time
    columns into a single datetime, reports duplicate count, and stores the raw
    event table in a dictionary.

    Parameters
    ----------
    input_file_list : list of str
        Ordered list of full filesystem paths to UX90-006 CSV export files.
    output_name_list : list of str
        Ordered list of output names (e.g. ``['living_room_pir']``) corresponding
        to each file in ``input_file_list``.
    skiprows : int
        Number of header rows to skip when reading each CSV.

    Returns
    -------
    occu_dict : dict
        Dictionary mapping each output name to a DataFrame of raw occupancy
        event timestamps (integer-indexed).
    """
    
    # Start of function
    occu_dict = {}    # Initialise a dictionary to accumulate occupancy DataFrames
    for input_files, output_name in zip(input_file_list, output_name_list):  # Pair each file path with its designated output name
        print('')  # Blank line for readability
        print('---------------------------------------------')  # Visual separator
        print('Ingesting ux90_006 data from', input_files)  # Log the source file
        temp = pd.read_csv(input_files, skiprows = skiprows, encoding='utf-8', engine='c')  # Read the occupancy event CSV with UTF-8 encoding and fast C parser
        temp.columns = ['date', 'time', output_name]  # Assign date, time, and occupancy-label column names
        temp['date'] = pd.to_datetime(temp['date'] + ' ' + temp['time'], errors='raise', dayfirst=True)  # Merge date and time strings into a single datetime
        del temp['time']  # Remove the now-redundant separate time column

        print('Non-unique observations = ', len(temp['date']) - temp['date'].nunique())  # Report how many timestamps are duplicated (potential sensor artefact indicator)
        # temp = temp.set_index(temp['date'])
        # temp = temp.resample('5T').bfill()
        # temp.index.names = ['index']
        # temp['date'] = temp.index
        #
        #
        #
        temp = temp.reset_index(drop=True)  # Reset to contiguous integer index, discarding the old index
        #
        #
        #
        print(temp)  # Display the ingested occupancy DataFrame for inspection
        occu_dict[output_name] = temp  # Store the event table in the dictionary under its output name
        print('ux90_006 data ingested')  # Confirm successful completion for this file
        print('---------------------------------------------')  # Visual separator
        print('')  # Trailing blank line
    return(occu_dict)  # Return the dictionary of occupancy event DataFrames

## co2_correction_unoccupied

Applies a piecewise-linear drift correction to indoor CO2 channels using unoccupied reference periods. NDIR CO2 sensors drift over months-long deployments; this function corrects for that drift by anchoring the correction to periods when the dwelling was unoccupied and ventilated, so indoor CO2 should equal outdoor CO2. For each pair of consecutive unoccupied periods the function: computes the indoor 10th-percentile and outdoor median CO2 during the period; calculates the offset (difference between the two percentiles); and applies a linearly interpolated correction between the midpoints of successive unoccupied periods. The first period correction is applied uniformly from the campaign start to the first midpoint; the last period offset is applied uniformly from the last midpoint to the campaign end. Users should verify that the identified unoccupied periods satisfy the method's assumptions before using corrected data.

In [ ]:
def co2_correction_unoccupied(  input_dataframe
                              , input_start_and_end_as_df
                              , input_start_date
                              , input_end_date
                              , input_list_unoccupied_periods
                              , input_outdoor_co2_col
                              , input_co2_cols_to_correct
                              , input_percentile_indoor
                              , input_percentile_outdoor
                              ):
    """
    Apply piecewise-linear NDIR CO2 drift correction anchored to unoccupied periods.

    For each indoor CO2 column, computes per-period offsets (indoor percentile
    minus outdoor percentile during each unoccupied period), linearly interpolates
    the correction between consecutive period midpoints, and applies it to the
    full campaign record. Modifies ``input_dataframe`` in place.

    Parameters
    ----------
    input_dataframe : pd.DataFrame
        Datetime-indexed DataFrame containing both indoor and outdoor CO2 columns.
    input_start_and_end_as_df : bool
        If ``True``, use the first and last index values as campaign start/end
        dates; otherwise use ``input_start_date`` and ``input_end_date``.
    input_start_date : str
        Campaign start datetime string (used when ``input_start_and_end_as_df``
        is ``False``).
    input_end_date : str
        Campaign end datetime string (used when ``input_start_and_end_as_df``
        is ``False``).
    input_list_unoccupied_periods : list of str
        Flat list of alternating start/end datetime strings for each unoccupied
        period, e.g. ``['2020-01-01 09:00:00', '2020-01-01 18:00:00', ...]``.
    input_outdoor_co2_col : str
        Column name of the outdoor CO2 reference channel.
    input_co2_cols_to_correct : list of str
        List of indoor CO2 column names to correct.
    input_percentile_indoor : float
        Percentile (0–100) used to characterise indoor CO2 during unoccupied
        periods (typically 10).
    input_percentile_outdoor : float
        Percentile (0–100) used to characterise outdoor CO2 during unoccupied
        periods (typically 50).

    Returns
    -------
    None
        The input DataFrame is modified in place; no object is returned.
    """

    print('')  # Blank line for readability
    print('============================================================')  # Top border of correction banner
    print('====    Start of CO2 correction    ====')  # Banner header
    print('============================================================')  # Bottom border of correction banner
    print('')  # Blank line after banner

    #### Determine start and end dates:
    # If input_start_and_end_as_df is True, then use the first and last index datetimes for start and end times for CO2 correction.
        
    if input_start_and_end_as_df == True:  # Use DataFrame bounds as campaign start and end
        start_date = input_dataframe.index[0]  # First timestamp in the DataFrame becomes the correction start
        end_date = input_dataframe.index[-1]  # Last timestamp in the DataFrame becomes the correction end
        print('')  # Blank line
        print('Using beginning and end dates from input df')  # Log that DataFrame bounds are being used
        print('Start date = ', start_date)  # Log the derived start date
        print('End date = ', end_date)  # Log the derived end date
        print('')  # Blank line
    
    else:  # Use caller-supplied date strings as campaign start and end
        start_date = input_start_date  # Use the caller-provided start date string
        end_date = input_end_date  # Use the caller-provided end date string
        print('')  # Blank line
        print('Using your input beginning and end dates')  # Log that caller-supplied dates are being used
        print('Start date = ', start_date)  # Log the start date
        print('End date = ', end_date)  # Log the end date
        print('')  # Blank line
    
    #### Main loop
    
    # For each co2 col
    # For each pair of dates
    # Calculate 
    # Loop counter - do different first and last
    
    for co2_col in input_co2_cols_to_correct:  # Outer loop: process each indoor CO2 column independently
        print('-------------------------------------------------------------')  # Visual separator for this CO2 column
        print('----     Correcting: ', co2_col,'    ----')  # Log the column currently being corrected
        print('-------------------------------------------------------------')  # Visual separator
        print('')  # Blank line
        #
        # For loop each pair of dates 
        #
        each_pair_dates_counter = 0  # Initialise loop counter to track first and last iterations
        #
        for unoccupied_start, unoccupied_end in zip(input_list_unoccupied_periods[0::2], input_list_unoccupied_periods[1::2]):  # Iterate over consecutive start/end pairs from the flat list
            # Loop counter
            each_pair_dates_counter = each_pair_dates_counter + 1  # Increment iteration counter
            # print('Loop number', each_pair_dates_counter)
            #
            # First loop
            #
            if each_pair_dates_counter == 1: # First loop
                print('Loop', each_pair_dates_counter)  # Log the loop number
                
                print('Start of unoccupied period', each_pair_dates_counter, '=', unoccupied_start)  # Log unoccupied period start
                print('End of unoccupied period', each_pair_dates_counter, '=', unoccupied_end)  # Log unoccupied period end
                #
                # Mid point between start and finish
                #
                from datetime import timedelta, datetime
                unoccupied_mid = datetime.strptime(unoccupied_start,"%Y-%m-%d %H:%M:%S") + (datetime.strptime(unoccupied_end,"%Y-%m-%d %H:%M:%S") - datetime.strptime(unoccupied_start,"%Y-%m-%d %H:%M:%S"))/2  # Compute the temporal midpoint of this unoccupied period
                print('Mid point =', unoccupied_mid)  # Log the computed midpoint
                import datetime
                #
                # Calculate the indoor and outdoor percentiles during the unoccupied period
                #
                indoor_slice = input_dataframe[co2_col].loc[unoccupied_start:unoccupied_end]  # Extract indoor CO2 slice for this unoccupied period
                outdoor_slice = input_dataframe[input_outdoor_co2_col].loc[unoccupied_start:unoccupied_end]  # Extract outdoor CO2 slice for this unoccupied period
                if indoor_slice.isna().all() or outdoor_slice.isna().all():  # Guard: skip if either slice is entirely NaN
                    print(f'warning: all-NaN slice for {co2_col} or {input_outdoor_co2_col} during {unoccupied_start} to {unoccupied_end} — skipping period')
                    continue
                indoor_percentile = np.nanpercentile(indoor_slice, input_percentile_indoor)  # Compute indoor CO2 percentile during the unoccupied period, ignoring NaN
                print('Indoor CO2', input_percentile_indoor , 'th percentile is ', indoor_percentile)  # Log the indoor percentile value
                #
                outdoor_percentile = np.nanpercentile(outdoor_slice, input_percentile_outdoor)  # Compute outdoor CO2 percentile during the same period
                print('Outdoor CO2', input_percentile_outdoor , 'th percentile is ', outdoor_percentile)  # Log the outdoor percentile value
                #
                offset = indoor_percentile - outdoor_percentile  # The drift offset is the excess of indoor over outdoor CO2 during unoccupied conditions
                print('Difference in CO2', input_percentile_indoor , 'th percentiles is ', offset)  # Log the computed offset
                #
                # CO2 correction
                #
                input_dataframe.loc[start_date:unoccupied_mid - datetime.timedelta(minutes=5), co2_col] = input_dataframe.loc[start_date:unoccupied_mid - datetime.timedelta(minutes=5), co2_col] - offset  # Apply constant offset correction from campaign start to just before the first midpoint
                # input_dataframe[co2_col].loc[start_date:unoccupied_mid- datetime.timedelta(minutes=5)] = input_dataframe[co2_col].loc[start_date:unoccupied_mid- datetime.timedelta(minutes=5)] - offset
                #
                print('')  # Blank line after first loop
            #
            # All other loops
            #
            else:
                print('Loop', each_pair_dates_counter) # All other loops
                print('Start of unoccupied period', each_pair_dates_counter, '=', unoccupied_start)  # Log unoccupied period start
                print('End of unoccupied period', each_pair_dates_counter, '=', unoccupied_end)  # Log unoccupied period end
                #
                # Mid point between start and finish
                #
                previous_unoccupied_mid = unoccupied_mid # Store previous mid
                #                
                from datetime import timedelta, datetime
                unoccupied_mid = datetime.strptime(unoccupied_start,"%Y-%m-%d %H:%M:%S") + (datetime.strptime(unoccupied_end,"%Y-%m-%d %H:%M:%S") - datetime.strptime(unoccupied_start,"%Y-%m-%d %H:%M:%S"))/2  # Compute midpoint of the current unoccupied period
                print('Mid point =', unoccupied_mid)  # Log the midpoint
                import datetime
                #
                # Length previous mid to current mid
                #
                length_previous_mid_to_current_mid = len(input_dataframe.loc[previous_unoccupied_mid:unoccupied_mid])  # Count the number of 5-minute rows between the previous and current midpoints
                print('Length between previous mid and this mid' , length_previous_mid_to_current_mid)  # Log the inter-midpoint row count
                #
                # Calculate the indoor and outdoor percentiles during the unoccupied period
                #
                indoor_slice = input_dataframe[co2_col].loc[unoccupied_start:unoccupied_end]  # Extract indoor CO2 slice for this unoccupied period
                outdoor_slice = input_dataframe[input_outdoor_co2_col].loc[unoccupied_start:unoccupied_end]  # Extract outdoor CO2 slice for this unoccupied period
                if indoor_slice.isna().all() or outdoor_slice.isna().all():  # Guard: skip if either slice is entirely NaN
                    print(f'warning: all-NaN slice for {co2_col} or {input_outdoor_co2_col} during {unoccupied_start} to {unoccupied_end} — skipping period')
                    continue
                indoor_percentile = np.nanpercentile(indoor_slice, input_percentile_indoor)  # Compute indoor CO2 percentile for the current period
                print('Indoor CO2', input_percentile_indoor , 'th percentile is ', indoor_percentile)  # Log the indoor percentile
                #
                outdoor_percentile = np.nanpercentile(outdoor_slice, input_percentile_outdoor)  # Compute outdoor CO2 percentile for the current period
                print('Outdoor CO2', input_percentile_outdoor , 'th percentile is ', outdoor_percentile)  # Log the outdoor percentile
                #
                # Calculate slope
                #
                previous_offset = offset # Store previous offset
                
                offset = indoor_percentile - outdoor_percentile  # Compute the drift offset for the current period
                print('Difference in CO2', input_percentile_indoor , 'th percentiles is ', offset)  # Log the current offset
                #
                # Difference in offsets
                
                offset_difference = offset - previous_offset  # Change in drift offset between consecutive periods drives the interpolated slope
                print('Offset difference = ', offset_difference)  # Log the offset difference
                #
                # Slope
                #
                slope = offset_difference/length_previous_mid_to_current_mid  # Slope of the linear correction ramp between consecutive midpoints (ppm per 5-min step)
                print('Slope = ', slope)  # Log the computed slope
                #
                # CO2 correction - offset
                #
                # input_dataframe[co2_col].loc[previous_unoccupied_mid + datetime.timedelta(minutes=5):unoccupied_mid - datetime.timedelta(minutes=5)] = input_dataframe[co2_col].loc[previous_unoccupied_mid + datetime.timedelta(minutes=5):unoccupied_mid - datetime.timedelta(minutes=5)] - previous_offset
                
                # Define the slice explicitly using .loc[]
                slice_start = previous_unoccupied_mid + datetime.timedelta(minutes=5)  # Start of this segment: one step after the previous midpoint
                slice_end = unoccupied_mid - datetime.timedelta(minutes=5)  # End of this segment: one step before the current midpoint
                
                # Modify the original DataFrame directly
                input_dataframe.loc[slice_start:slice_end, co2_col] = (
                    input_dataframe.loc[slice_start:slice_end, co2_col] - previous_offset)  # Apply the previous period's constant offset to the inter-midpoint segment
                #
                # CO2 correction - slope
                #
                slope_multiplier = 0  # Initialise row counter for the linear slope correction
                for index, row in input_dataframe.loc[previous_unoccupied_mid:unoccupied_mid].iterrows(): # loop over two rows at a time
                    slope_multiplier = slope_multiplier + 1  # Increment the row counter for each 5-minute step
                    input_dataframe.loc[index, co2_col] = input_dataframe.loc[index, co2_col] - (slope * slope_multiplier)  # Subtract the linearly increasing slope correction at each step
                print('')  # Blank line after each non-first loop
            #
            # Last loop
            #
            if each_pair_dates_counter == (len(input_list_unoccupied_periods))/2: # Last loop
                print('Last loop')  # Log that this is the final unoccupied period
                print('Loop', each_pair_dates_counter)  # Log the loop number
                
                print('Start of unoccupied period', each_pair_dates_counter, '=', unoccupied_start)  # Log unoccupied period start
                print('End of unoccupied period', each_pair_dates_counter, '=', unoccupied_end)  # Log unoccupied period end
                #
                #
                # Calculate the indoor and outdoor percentiles during the unoccupied period
                #
                indoor_slice = input_dataframe[co2_col].loc[unoccupied_start:unoccupied_end]  # Extract indoor CO2 slice for this unoccupied period
                outdoor_slice = input_dataframe[input_outdoor_co2_col].loc[unoccupied_start:unoccupied_end]  # Extract outdoor CO2 slice for this unoccupied period
                if indoor_slice.isna().all() or outdoor_slice.isna().all():  # Guard: skip if either slice is entirely NaN
                    print(f'warning: all-NaN slice for {co2_col} or {input_outdoor_co2_col} during {unoccupied_start} to {unoccupied_end} — skipping period')
                    continue
                indoor_percentile = np.nanpercentile(indoor_slice, input_percentile_indoor)  # Compute indoor CO2 percentile for the final period
                print('Indoor CO2', input_percentile_indoor , 'th percentile is ', indoor_percentile)  # Log the indoor percentile
                #
                outdoor_percentile = np.nanpercentile(outdoor_slice, input_percentile_outdoor)  # Compute outdoor CO2 percentile for the final period
                print('Outdoor CO2', input_percentile_outdoor , 'th percentile is ', outdoor_percentile)  # Log the outdoor percentile
                #
                previous_offset = offset # Store previous offset                
                #
                offset = indoor_percentile - outdoor_percentile  # Compute the drift offset for the final period
                print('Difference in CO2', input_percentile_indoor , 'th percentiles is ', offset)  # Log the final offset
                #
                # CO2 correction
                #
                input_dataframe.loc[unoccupied_mid + datetime.timedelta(minutes=5):end_date, co2_col] -= previous_offset  # Apply the penultimate offset uniformly from the last midpoint to the end of the campaign
                #
                print('')  # Blank line after last loop
    
    print('')  # Blank line before closing banner
    print('============================================================')  # Top border of closing banner
    print('====    End of CO2 correction    ====')  # Closing banner
    print('============================================================')  # Bottom border
    print('')  # Trailing blank line

## co2_col_list

Returns a list of all indoor CO2 column names from a DataFrame. Iterates over every column and includes those whose name contains `'co2'` but excludes columns whose name also contains `'external'` (the outdoor reference channel). This convenience function is used to obtain the set of indoor CO2 channels to pass to `co2_correction_unoccupied` without having to manually enumerate column names for each case study.

In [ ]:
def co2_col_list(input_df):
    """
    Return a list of all indoor CO2 column names from a DataFrame.

    Includes columns whose name contains ``'co2'`` but excludes those that
    also contain ``'external'`` (the outdoor reference channel).

    Parameters
    ----------
    input_df : pd.DataFrame
        DataFrame whose column names are scanned for CO2 channels.

    Returns
    -------
    co2_col_list : list of str
        List of indoor CO2 column name strings.
    """
    co2_col_list = []  # Initialise empty list to accumulate indoor CO2 column names
    for column in input_df.columns:  # Iterate over all column names in the DataFrame
        if 'co2' in column:  # Include only columns that contain 'co2' in their name
            if 'external' in column:  # Exclude the outdoor (external) CO2 reference column
                print('')  # Skip silently — external CO2 is not corrected by this function
            else:
            #print(column)
                co2_col_list += (str(column),)  # Append the indoor CO2 column name as a string
    return(co2_col_list)  # Return the list of indoor CO2 column names

## ingest_ceda_weather_data

Ingests hourly weather data from a CEDA Met Office CSV and resamples to 5-minute resolution. CEDA exports contain an `ob_time` column parsed directly by pandas and numerous meteorological variables; the caller specifies which columns to retain and their new names. The function replaces space-only strings with NaN equivalents, applies backward fill to remove NaN values (typical for CEDA's missing-data encoding), groups by timestamp to handle any duplicates arising from daylight-saving transitions, then forward-fills and backward-fills to a uniform 5-minute frequency. The result is a continuous 5-minute weather DataFrame suitable for joining with indoor sensor data.

In [ ]:
def ingest_ceda_weather_data(input_path
                             , skiprows
                             , cols_from_ceda
                             , new_col_names
                             ):
    """
    Ingest and resample CEDA Met Office hourly weather data to 5-minute resolution.

    Reads a CEDA CSV, selects and renames columns, parses the datetime index,
    replaces missing-data strings, back-fills NaN values, aggregates duplicate
    timestamps, and resamples to 5-minute intervals via forward and backward fill.

    Parameters
    ----------
    input_path : str
        Full filesystem path to the CEDA Met Office CSV file.
    skiprows : int
        Number of header rows to skip when reading the CSV.
    cols_from_ceda : list of str
        Column names (as they appear in the raw CEDA CSV) to retain.
    new_col_names : list of str
        New column names to assign, corresponding positionally to
        ``cols_from_ceda``; must include ``'date'`` as the first entry.

    Returns
    -------
    df_tmp : pd.DataFrame
        Datetime-indexed DataFrame at 5-minute resolution containing the
        selected and renamed meteorological variables.
    """
    
    # Start of function
    print('')  # Blank line for readability
    print('---------------------------------------------')  # Visual separator
    print('Ingesting weather data from', input_path)  # Log the source file
    df_tmp= pd.read_csv(input_path, skiprows = skiprows, encoding='utf-8', engine='c', low_memory=False, parse_dates=['ob_time'], dayfirst=True)  # Read the CEDA CSV, parsing ob_time directly and using the fast C parser
    
    df_tmp = df_tmp.replace(" ", "")  # Replace whitespace-only strings with empty string to normalise missing-data encoding
    df_tmp = df_tmp[cols_from_ceda].copy()  # Select only the requested CEDA columns (copy to avoid SettingWithCopyWarning)
    df_tmp.columns = new_col_names  # Apply caller-supplied column names
    
    df_tmp['date'] = pd.to_datetime(df_tmp['date'], errors='raise', dayfirst=False)  # Parse the date column as datetime (CEDA timestamps are ISO format, not day-first)
    df_tmp = df_tmp.set_index(df_tmp['date'])  # Set the datetime column as the DataFrame index
    del df_tmp['date']  # Remove the now-redundant 'date' column
    df_tmp = df_tmp.apply(pd.to_numeric)  # Convert all columns to numeric, coercing non-numeric strings to NaN
    
    print('nans before interpolating = ',df_tmp.isna().sum()) # how many nans?
    # df_tmp = df_tmp.interpolate(method ='bfill', limit_direction ='backward')
    df_tmp = df_tmp.bfill()  # Back-fill to remove NaN values arising from CEDA's missing-data encoding
    
    print('nans before interpolating = ',df_tmp.isna().sum()) #Should equal 0
    df_tmp = df_tmp.groupby(['date'], as_index=True).mean()  # Average any duplicate timestamps (e.g. from daylight-saving transitions)
    df_tmp = df_tmp.resample('5min').ffill().bfill()  # resample, forward fill, and backfill for 5 minutes
    print('Weather data ingested')  # Confirm successful completion
    
    return(df_tmp)  # Return the resampled 5-minute weather DataFrame

## rooms_variables_dict

Builds a dictionary mapping each room name to the list of variables measured in that room. The function infers room names by stripping the variable suffix from column names that match any of the supplied variable types (e.g. `'temperature'`, `'co2'`). It then iterates over the unique room names and collects all column suffixes for each room, excluding `'window'` and `'door'` columns (which are event sensors, not continuous measurements). The result is used for grouped plotting and room-level statistical comparisons.

In [ ]:
def rooms_variables_dict(  input_dataframe
                         , variables):
    """
    Build a dictionary mapping each room name to its measured variable list.

    Infers room names from column names by stripping known variable suffixes,
    then collects all non-event variable suffixes for each room.

    Parameters
    ----------
    input_dataframe : pd.DataFrame
        DataFrame whose column names encode both room and variable information
        in the format ``'<room>_<variable>'``.
    variables : list of str
        Variable name fragments used to identify room columns, e.g.
        ``['temperature', 'humidity', 'co2', 'pm25']``.

    Returns
    -------
    dictionary_temp : dict
        Dictionary where each key is a room name (str) and each value is a
        list of variable name strings measured in that room.
    """
    
    # Loop over all columns and get a list of all rooms
    # Create a dictionary with a df for each rooms and the variables in that rooms
    
    rooms = [] # instantiate empty list
    dictionary_temp = {} # instantiate dict
    
    for col in input_dataframe:  # Iterate over all column names
        for variable in variables:  # Check each candidate variable suffix
            if variable in col:  # If this column contains the variable name, extract the room prefix
                rooms.append(col.replace('_' + variable,''))  # Strip the variable suffix to obtain the room name
    
    rooms = set(rooms)  # Deduplicate room names (a room will appear once per variable it hosts)
    rooms = list(rooms)  # Convert back to a list for ordered iteration
    
    # print(rooms)
    for room in rooms:  # Iterate over each unique room name
        variable_list_temp = []  # Initialise the variable list for this room
        print(room)  # Log the room name being processed
        for col in input_dataframe:  # Scan all columns for variables belonging to this room
            if 'window' in col:  # Skip window event-sensor columns — not continuous measurements
                print('')  # Silent skip
            elif 'door' in col:  # Skip door event-sensor columns — not continuous measurements
                print('')  # Silent skip
            elif room in col:  # This column belongs to the current room
                variable_list_temp.append(col.replace(room+'_',''))  # Strip the room prefix to get the variable name
        print(variable_list_temp)  # Log the variables collected for this room
        dictionary_temp[room] = variable_list_temp  # Store the variable list under the room name key
    
    return(dictionary_temp)  # Return the completed room-to-variables dictionary

## event_durations

Calculates the time elapsed between consecutive events in each event DataFrame within an event dictionary. For each DataFrame, the function resets the integer index (required for sequential `.loc` access), adds a `'mins_next_event'` column, and iterates row by row to compute the difference in minutes between successive `'date'` values using `total_seconds() / 60`. The last row in each DataFrame gets a duration of zero (no subsequent event). The modified dictionary is returned with the new column in place.

In [ ]:
def event_durations(input_event_dict):
    """
    Compute elapsed minutes between consecutive events in each event DataFrame.

    For each DataFrame in the event dictionary, resets the index, adds a
    ``'mins_next_event'`` column, and fills it by computing the difference
    in minutes between each row and the next.

    Parameters
    ----------
    input_event_dict : dict
        Dictionary mapping event names to DataFrames that each contain a
        ``'date'`` column of datetime values.

    Returns
    -------
    input_event_dict : dict
        The same dictionary with a ``'mins_next_event'`` (float) column added
        to each DataFrame. The last row of each DataFrame has a duration of 0.
    """
    for dataframe in input_event_dict:  # Iterate over each event name in the dictionary
        # print('==============================================')
        print(dataframe)  # Log the event name being processed
        
        df = input_event_dict[dataframe].reset_index(drop=True)  # reset df_tmp index
        df['mins_next_event'] = 0.0  # Initialise the duration column to zero for all rows (float to accommodate fractional minutes)

        
        for index, row in df.iterrows():  # Iterate over each row
            # print(index, row)
            # print('----------------------------------------------')
            # print(dataframe)
            # print(index)
            if index + 1 == len(df):  # End
                print('')  # Last row: no subsequent event so duration stays 0; skip silently
            else:
                df.loc[index, 'mins_next_event'] = (
                    (df.loc[index + 1, 'date'] - df.loc[index, 'date']).total_seconds() / 60
                )  # Compute elapsed minutes to the next event using timedelta.total_seconds()
        input_event_dict[dataframe] = df
    
    return input_event_dict  # Return the dictionary with duration columns populated


## event_list_function

Returns a flat list of all event names from an event dictionary. This is a simple convenience function that iterates over the dictionary keys, logs each key to the console, and returns them as a list of strings. The result is typically used to programmatically generate plot titles, axis labels, or to pass the event names to other functions that require an explicit list.

In [ ]:
def event_list_function(input_event_dict):
    """
    Return a flat list of all event names from an event dictionary.

    Parameters
    ----------
    input_event_dict : dict
        Dictionary mapping event names (str) to event DataFrames.

    Returns
    -------
    event_list : list of str
        Ordered list of all event name strings (dictionary keys).
    """
    event_list = []  # Initialise empty list to accumulate event names
    for event in input_event_dict:  # Iterate over dictionary keys
        print(event)  # Log each event name for inspection
        event_list.append(str(event))  # Append the key as a string to the list
    return(event_list)  # Return the complete list of event names

## sum_sound_calculation_df

Computes an energy-averaged (logarithmic) sound level from multiple one-third-octave band columns. Sound levels in dB cannot be averaged arithmetically; instead, each dB value must be converted to a linear power ratio (10^(L/10)), summed, divided by the number of bands, and then converted back to dB (10 × log10). This function implements that standard logarithmic averaging procedure, as described by Cirrus Research, on a DataFrame column-by-column, returning a single `'calc'` Series of energy-averaged levels for each row.

In [ ]:
"""
https://www.cirrusresearch.co.uk/blog/2013/01/noise-data-averaging-how-do-i-average-noise-measurements/
Divide number by 10
Exponent
Sum all
Divide by number of obs
Take log (base 10)
Multiply by 10
"""

def sum_sound_calculation_df(input_df, input_col_list):
    """
    Compute an energy-averaged sound level across one-third-octave band columns.

    Converts each dB column to a linear power ratio (10^(L/10)), sums across
    the specified columns, divides by the number of columns to obtain the mean
    power, and converts back to dB (10 * log10). Implements the standard
    logarithmic averaging procedure for noise measurements.

    Parameters
    ----------
    input_df : pd.DataFrame
        DataFrame containing one-third-octave band sound level columns in dB.
    input_col_list : list of str
        Names of the columns in ``input_df`` to include in the averaging.

    Returns
    -------
    temp_df['calc'] : pd.Series
        Series of energy-averaged sound levels (dB) for each row of the input.
    """
    temp_df = input_df.copy()  # Work on a copy to avoid modifying the caller's DataFrame
    for col in input_col_list:  # Convert each specified dB column to a linear power ratio
        temp_df[col] = 10**(temp_df[col]/10)  # Apply the dB-to-linear conversion: power = 10^(L/10)
    temp_df['calc'] = temp_df[input_col_list].sum(axis=1, min_count=1)  # Sum linear power ratios; min_count=1 returns NaN (not 0) when all bands are NaN, avoiding log10(0)
    temp_df['calc'] = 10*(np.log10(temp_df['calc']/len(input_col_list)))  # Divide by the number of bands to get mean power, then convert back to dB
    return(temp_df['calc'])  # Return the Series of energy-averaged dB levels

## rpi_noise_ingest

Ingests one-third-octave noise CSV files from Raspberry Pi PHEUCLio units and returns structured DataFrames. The PHEUCLio platform writes daily CSV files named with the convention `<unit>TOBin.csv` (indoor microphone) or `<unit>TOBout.csv` (outdoor microphone), each containing 30 columns: timestamp, Leq, LAeq, and 28 one-third-octave band levels from 25 Hz to 10 kHz. The function walks the directory tree from `input_start_directory`, identifies all `TOB`-containing CSVs, parses the unit name from the directory path, concatenates all files into a single DataFrame, removes duplicate rows, applies hardcoded LAeq corrections for two specific units (PHEUCLio009 outdoor +3 dB; PHEUCLio012 outdoor −0.5 dB) derived from the post-campaign colocation, builds a dictionary keyed by `(unit, mic)` tuples, and returns separate indoor and outdoor DataFrames resampled to 5-minute resolution.

In [ ]:
def rpi_noise_ingest(input_start_directory):
    """
    Ingest one-third-octave noise CSV files from Raspberry Pi PHEUCLio units.

    Walks the directory tree from ``input_start_directory``, finds all CSV files
    whose names contain ``'TOB'``, parses the unit name and microphone location
    (indoor/outdoor) from the path and filename, concatenates into a single
    DataFrame, removes duplicates, applies per-unit LAeq corrections, builds a
    ``(unit, mic)``-keyed dictionary, and returns separate 5-minute-resampled
    indoor and outdoor DataFrames.

    Parameters
    ----------
    input_start_directory : str
        Root directory path from which to begin the recursive CSV search.

    Returns
    -------
    noise_dict : dict
        Dictionary keyed by ``(unit, mic)`` tuples mapping to DataFrames of
        raw noise data for each unit and microphone combination.
    noise_inside : pd.DataFrame
        Datetime-indexed DataFrame of indoor LAeq columns resampled to 5 minutes
        via forward fill.
    noise_outside : pd.DataFrame
        Datetime-indexed DataFrame of outdoor LAeq columns resampled to 5 minutes
        via forward fill.
    """
    
    # Constants
    START_DIRECTORY = input_start_directory  # Directory containing the noise data
    CSV_COLUMNS = [
        'date', 'leq', 'laeq', '25hz', '31.5hz', '40hz', '50hz', '63hz', '80hz', '100hz',
        '125hz', '160hz', '200hz', '250hz', '315hz', '400hz', '500hz', '630hz', '800hz',
        '1000hz', '1250hz', '1600hz', '2000hz', '2500hz', '3150hz', '4000hz', '5000hz',
        '6300hz', '8000hz', '10000hz'
    ]  # Column names matching the 30-column PHEUCLio TOB CSV format
    
    # Initialize the result DataFrame
    df_result = pd.DataFrame()  # Initialise empty accumulator DataFrame for all files
    
    # Walk through the directory to find and process CSV files
    for path, dirs, files in os.walk(START_DIRECTORY):  # Recursively traverse all subdirectories
        for file in fnmatch.filter(files, '*.csv'):  # Only process CSV files
            if 'TOB' in file:  # Check if the file name contains 'TOB'
                # Parse file name to determine microphone location
                before, sep, after = file.rpartition("TOB")  # Split on 'TOB' to extract the suffix
                mic_location = after.replace('.csv', '')  # Example: 'in' or 'out'
    
                # Extract noise unit from the directory path
                before, sep, after = path.rpartition("noise/")  # Split on 'noise/' to isolate the unit folder
                noise_unit = after  # Example: 'PHEUCLio009'
    
                # Construct full file path
                full_file_path = os.path.join(path, file)  # Combine directory path and filename into a full path
    
                try:
                    # Read the CSV file into a DataFrame
                    df_temp = pd.read_csv(full_file_path, header=None, names=CSV_COLUMNS)  # Read without header row, applying the predefined column names
    
                    # Convert 'date' column to datetime and set it as the index
                    df_temp['date'] = pd.to_datetime(df_temp['date'], errors='raise', dayfirst=True)  # Parse the timestamp column as datetime
                    df_temp.set_index('date', inplace=True)  # Set datetime as the index for time-series operations
    
                    # Add metadata columns for noise unit and microphone location
                    df_temp['unit'] = noise_unit  # Tag each row with the PHEUCLio unit identifier
                    df_temp['mic'] = mic_location  # Tag each row with the microphone location (in/out)
    
                    # Append the processed data to the result DataFrame
                    df_result = pd.concat([df_result, df_temp], axis='index', ignore_index=False)  # Concatenate this file's data to the accumulator
    
                except Exception as e:
                    print(f"Error processing file {file}: {e}")  # Log any file-level read errors without halting the loop
    
    # Remove duplicates based on 'date', 'unit', and 'mic'
    df_result_reset = df_result.reset_index()  # Temporarily reset index to access 'date' as a column
    df_result_cleaned = df_result_reset[~df_result_reset.duplicated(subset=['date', 'unit', 'mic'], keep='first')]  # Keep only the first occurrence of each (date, unit, mic) combination
    df_result_cleaned = df_result_cleaned.set_index('date')  # Restore 'date' as the index
    
    # Apply corrections to 'laeq' for specific units and microphones
    df_result.loc[(df_result['unit'] == 'PHEUCLio009') & (df_result['mic'] == 'out'), 'laeq'] += 3  # Apply +3 dB correction to PHEUCLio009 outdoor channel from post-campaign colocation
    df_result.loc[(df_result['unit'] == 'PHEUCLio012') & (df_result['mic'] == 'out'), 'laeq'] -= 0.5  # Apply -0.5 dB correction to PHEUCLio012 outdoor channel from post-campaign colocation
    
    # Create a dictionary of DataFrames grouped by noise unit and microphone location
    units = df_result['unit'].unique()  # Get the list of unique PHEUCLio unit identifiers present in the data
    mics = df_result['mic'].unique()  # Get the list of unique microphone location codes (in/out)
    noise_dict = {
        (unit, mic): df_result[(df_result['unit'] == unit) & (df_result['mic'] == mic)]
        for unit in units for mic in mics
    }  # Build a dictionary keyed by (unit, mic) tuples containing the corresponding sub-DataFrames
    
    # Create a combined DataFrame for noise levels
    noise = pd.DataFrame()  # Initialise empty DataFrame to hold combined LAeq columns
    for (unit, mic), df in noise_dict.items():  # Iterate over each (unit, mic) combination
        col_name = f"{unit}_{mic}"  # Column name combines unit and microphone location
        noise = noise.join(df['laeq'].rename(col_name), how='outer')  # Join this unit's LAeq series as a new column, preserving all timestamps
    
    # Remove duplicate indices in the combined noise DataFrame
    noise = noise.loc[~noise.index.duplicated(keep='first')]  # Drop duplicate datetime index entries, keeping the first occurrence
    
    # Resample noise data to 5-minute intervals for indoor and outdoor locations
    noise_inside = noise.filter(like='in').resample('5min').ffill()  # Select indoor columns and forward-fill to 5-minute resolution
    noise_outside = noise.filter(like='out').resample('5min').ffill()  # Select outdoor columns and forward-fill to 5-minute resolution
    
    return noise_dict, noise_inside, noise_outside  # Return the unit-mic dictionary and the two resampled summary DataFrames

## summary_statistics

Computes summary statistics for all continuous measurements and events across a case-study dwelling. For each room–variable combination in `input_rooms_and_variables`, the function reports the count of valid observations, count of missing values, mean, standard deviation, minimum, 5th, 25th, 50th, 75th, 95th percentile, and maximum from the main time-series DataFrame. For each entry in `input_event_dict`, it computes the total number of state transitions, the fraction of campaign time the sensor was in the open state (from the inverted binary column in the main DataFrame), and open-period duration statistics (mean, median, p5, p25, p75, p95, max in minutes) derived from the `mins_next_event` column produced by `event_durations`. The room for each event is looked up from `input_rooms_and_events`. Two DataFrames are returned: `continuous_stats` and `event_stats`.

In [ ]:
def summary_statistics(input_dataframe
                       , input_rooms_and_variables
                       , input_event_dict
                       , input_rooms_and_events
                       ):
    """
    Compute summary statistics for all continuous measurements and events.

    For each room–variable combination in ``input_rooms_and_variables``, computes
    descriptive statistics (n, n_missing, mean, std, min, p5, p25, median, p75,
    p95, max) from the main time-series DataFrame. For each event in
    ``input_event_dict``, computes the number of transitions, fraction of time
    open, and open-period duration statistics (mean, median, p5, p25, p75, p95,
    max in minutes) using the ``mins_next_event`` column produced by
    ``event_durations``. In the raw event DataFrame, a value of 0 represents
    the sensor transitioning to the open state and 1 represents closed; the
    function uses these raw values to identify open-period durations.

    Parameters
    ----------
    input_dataframe : pd.DataFrame
        Main datetime-indexed DataFrame containing all continuous measurements
        and binary event state columns (1 = open, 0 = closed after inversion).
    input_rooms_and_variables : dict
        Dictionary mapping room name strings to lists of variable name strings,
        as returned by ``rooms_variables_dict``.
    input_event_dict : dict
        Dictionary mapping event name strings to DataFrames of raw event
        records, each containing the event-name column (0/1 state) and a
        ``mins_next_event`` column, as produced by ``ingest_ux90_001m_group_new``
        and ``event_durations``.
    input_rooms_and_events : dict
        Dictionary mapping room name strings to lists of event name strings
        belonging to that room, used to annotate the event stats output.

    Returns
    -------
    continuous_stats : pd.DataFrame
        One row per room–variable combination with columns: room, variable,
        column, n, n_missing, mean, std, min, p5, p25, median, p75, p95, max.
    event_stats : pd.DataFrame
        One row per event with columns: room, event, n_transitions,
        n_open_periods, fraction_time_open, open_mean_mins, open_median_mins,
        open_p5_mins, open_p25_mins, open_p75_mins, open_p95_mins, open_max_mins.
    """

    print('')  # Blank line for readability
    print('============================================================')  # Top border
    print('====    Summary Statistics    ====')  # Banner header
    print('============================================================')  # Bottom border
    print('')  # Blank line after banner

    # build a reverse lookup: event name -> room, for annotating event_stats
    event_to_room = {}  # Initialise empty lookup dict
    for room, events in input_rooms_and_events.items():  # Iterate over each room and its event list
        for event in events:  # Iterate over each event in the room
            if event not in event_to_room:  # Only record the first room that claims this event
                event_to_room[event] = room  # Map event name to its room

    # -----------------------------------------------------------------
    # continuous variable statistics
    # -----------------------------------------------------------------
    print('--- Continuous Variables ---')  # Section header
    print('')  # Blank line
    continuous_rows = []  # Accumulate one dict per variable

    for room in input_rooms_and_variables:  # Iterate over rooms
        for variable in input_rooms_and_variables[room]:  # Iterate over variables in this room
            col = room + '_' + variable  # Construct the column name
            if col not in input_dataframe.columns:  # Skip if the column is absent from the DataFrame
                continue
            series = input_dataframe[col].dropna()  # Drop NaN values for percentile calculations
            n_total = len(input_dataframe[col])  # Total number of rows (including NaN)
            n_valid = len(series)  # Number of non-NaN observations
            n_missing = n_total - n_valid  # Number of missing observations
            if n_valid == 0:  # All-NaN column: fill stats with NaN
                row = {
                    'room': room, 'variable': variable, 'column': col,
                    'n': 0, 'n_missing': n_missing,
                    'mean': np.nan, 'std': np.nan,
                    'min': np.nan, 'p5': np.nan, 'p25': np.nan,
                    'median': np.nan, 'p75': np.nan, 'p95': np.nan, 'max': np.nan,
                }
            else:
                row = {
                    'room': room,  # Room name
                    'variable': variable,  # Variable name
                    'column': col,  # Full column name
                    'n': n_valid,  # Count of valid observations
                    'n_missing': n_missing,  # Count of missing observations
                    'mean': round(float(np.mean(series)), 3),  # Arithmetic mean
                    'std': round(float(np.std(series)), 3),  # Standard deviation
                    'min': round(float(np.min(series)), 3),  # Minimum value
                    'p5': round(float(np.percentile(series, 5)), 3),  # 5th percentile
                    'p25': round(float(np.percentile(series, 25)), 3),  # 25th percentile
                    'median': round(float(np.median(series)), 3),  # Median (50th percentile)
                    'p75': round(float(np.percentile(series, 75)), 3),  # 75th percentile
                    'p95': round(float(np.percentile(series, 95)), 3),  # 95th percentile
                    'max': round(float(np.max(series)), 3),  # Maximum value
                }
            continuous_rows.append(row)  # Accumulate this row
            print(f'  {col}: n={n_valid}, n_missing={n_missing}, mean={row["mean"]}, median={row["median"]}')  # Log one-line summary

    continuous_stats = pd.DataFrame(continuous_rows)  # Assemble all rows into a DataFrame
    print('')  # Blank line before events section

    # -----------------------------------------------------------------
    # event statistics
    # -----------------------------------------------------------------
    print('--- Events ---')  # Section header
    print('')  # Blank line
    event_rows = []  # Accumulate one dict per event

    for event_name, event_df in input_event_dict.items():  # Iterate over each event
        room = event_to_room.get(event_name, 'unknown')  # Look up the room for this event

        # fraction of time open from the main DataFrame (1 = open after inversion)
        if event_name in input_dataframe.columns:  # Only if the event column was back-filled into the main df
            col_series = input_dataframe[event_name]  # Extract the binary open/closed column
            fraction_open = round(float(col_series.mean()), 3) if col_series.notna().any() else np.nan  # Mean of 1/0 gives fraction open
        else:
            fraction_open = np.nan  # Event not present in main DataFrame

        # open-period duration statistics from the raw event DataFrame
        # raw event values: 0 = transitioned to open, 1 = transitioned to closed
        # mins_next_event = duration of the current state before the next transition
        # exclude rows where mins_next_event == 0 (sentinel value set for the last row)
        if event_name in event_df.columns and 'mins_next_event' in event_df.columns:  # Standard event DataFrame structure present
            valid_durations = event_df[event_df['mins_next_event'] > 0]  # Exclude the last-row sentinel and any zero-duration artefacts
            open_durations = valid_durations.loc[valid_durations[event_name] == 0, 'mins_next_event']  # Rows where sensor transitioned to open state
            n_transitions = len(event_df)  # Total number of state transitions recorded
            n_open_periods = len(open_durations)  # Number of distinct open periods

            if n_open_periods > 0:  # At least one open period: compute full duration stats
                row = {
                    'room': room,  # Room this event belongs to
                    'event': event_name,  # Event name
                    'n_transitions': n_transitions,  # Total state changes
                    'n_open_periods': n_open_periods,  # Number of times opened
                    'fraction_time_open': fraction_open,  # Fraction of campaign time in open state
                    'open_mean_mins': round(float(open_durations.mean()), 1),  # Mean open duration (minutes)
                    'open_median_mins': round(float(open_durations.median()), 1),  # Median open duration (minutes)
                    'open_p5_mins': round(float(np.percentile(open_durations, 5)), 1),  # 5th percentile open duration
                    'open_p25_mins': round(float(np.percentile(open_durations, 25)), 1),  # 25th percentile open duration
                    'open_p75_mins': round(float(np.percentile(open_durations, 75)), 1),  # 75th percentile open duration
                    'open_p95_mins': round(float(np.percentile(open_durations, 95)), 1),  # 95th percentile open duration
                    'open_max_mins': round(float(open_durations.max()), 1),  # Maximum open duration (minutes)
                }
            else:  # No open periods recorded (e.g. sensor never opened or file missing)
                row = {
                    'room': room, 'event': event_name,
                    'n_transitions': n_transitions, 'n_open_periods': 0,
                    'fraction_time_open': fraction_open,
                    'open_mean_mins': np.nan, 'open_median_mins': np.nan,
                    'open_p5_mins': np.nan, 'open_p25_mins': np.nan,
                    'open_p75_mins': np.nan, 'open_p95_mins': np.nan,
                    'open_max_mins': np.nan,
                }
        else:  # Unexpected DataFrame structure: return minimal row
            row = {
                'room': room, 'event': event_name,
                'n_transitions': len(event_df), 'n_open_periods': np.nan,
                'fraction_time_open': fraction_open,
                'open_mean_mins': np.nan, 'open_median_mins': np.nan,
                'open_p5_mins': np.nan, 'open_p25_mins': np.nan,
                'open_p75_mins': np.nan, 'open_p95_mins': np.nan,
                'open_max_mins': np.nan,
            }

        event_rows.append(row)  # Accumulate this row
        print(f'  {event_name} ({room}): n_transitions={row["n_transitions"]}, fraction_open={fraction_open}, open_median_mins={row["open_median_mins"]}')  # Log one-line summary

    event_stats = pd.DataFrame(event_rows)  # Assemble all rows into a DataFrame
    print('')  # Blank line before closing banner
    print('============================================================')  # Top border of closing banner
    print('====    End of Summary Statistics    ====')  # Closing banner
    print('============================================================')  # Bottom border
    print('')  # Trailing blank line

    return continuous_stats, event_stats  # Return both summary DataFrames

## save_participant_outputs

In [ ]:
def save_participant_outputs(n, df, event_dict, rooms_and_variables,
                             rooms_and_events, rooms_list,
                             continuous_stats, event_stats, data_dir=".."):
    """save participant outputs to pickle for cross-participant analysis."""
    os.makedirs(f'{data_dir}/data_processed', exist_ok=True)
    pkl_path = f'{data_dir}/data_processed/p{n}_outputs.pkl'
    with open(pkl_path, 'wb') as f:
        pickle.dump({
            f'p{n}': df,
            f'p{n}_event_dict': event_dict,
            f'p{n}_rooms_and_variables': rooms_and_variables,
            f'p{n}_rooms_and_events': rooms_and_events,
            f'p{n}_rooms_list': rooms_list,
            f'p{n}_continuous_stats': continuous_stats,
            f'p{n}_event_stats': event_stats,
        }, f)
    print(f'p{n} outputs saved to {pkl_path}')


## The end

In [ ]:
print('')  # Blank line to separate function definitions from any subsequent code
print('All functions have been defined.')  # Confirmation message

